In [ ]:
!pip install pandas tqdm

In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm

# --- Configuration ---
# To make authenticated requests, generate a personal access token (PAT)
# from your GitHub settings > Developer settings > Personal access tokens.
# This increases your API rate limit from 60 to 5,000 requests per hour.
access_token = ''
headers = {'Authorization': f'token {access_token}'} if access_token else {}

# --- NEW: Language Filter ---
# Define the programming languages you want to target.
# The script will only save repositories where the primary language matches one in this list.
# The comparison is case-insensitive.
# Add `None` to this list if you want to include repositories that don't have a language specified.
target_languages = [
    'Java', 'C', 'C++', 'JavaScript', 'Python', 'C#', 'TypeScript'
]
# Convert to a set of lowercase strings for efficient, case-insensitive lookup
target_languages_set = {lang.lower() for lang in target_languages if lang}
if None in target_languages:
    target_languages_set.add(None)


# List of foundations and organizations and their associated GitHub paths
org_urls = {
    'Apache': ['apache'],
    'Mozilla': ['mozilla'],
    'GNU': ['gnu-org', 'gnu'],
    'Linux Foundation': ['linuxfoundation'],
    'Oniro': ['oniroproject'],
    'RailCasts': ['railscasts'], # This is a user account
    'Cloud Native Computing Foundation': ['cncf'],
    'GNOME': ['GNOME'],
    'KDE': ['KDE'],
    'OpenStack': ['openstack'],
    'Open Source Geospatial Foundation': ['OSGeo'],
    'Software Heritage': ['SoftwareHeritage'],
    'Open Knowledge Foundation': ['okfn'],
    'Wikimedia': ['wikimedia'],
    'OurResearch': ['ourresearch'],
    'Berkeley': ['berkeley-cs', 'ucb-bar', 'amplab'],
    'MIT': ['mit', 'mit-pdos', 'mit-csail'], # 'mit' is a user, others are orgs
    'Stanford University': ['stanford-cs', 'sul-dlss'],
    'Audiopedia Foundation': ['audiopedia'],
    'Open Source Security Foundation': ['ossf'],
    'OpenJS Foundation': ['openjs-foundation'],
    'Academy Software Foundation': ['AcademySoftwareFoundation'],
    'Open Mobility Foundation': ['openmobilityfoundation'],
    'OSU Open Source Lab': ['osuosl'],
    'FOSSi Foundation': ['fossi-foundation'],
    'OpenWallet Foundation': ['openwallet-foundation'],
    'veraPDF': ['veraPDF'],
    'Nomic Foundation': ['NomicFoundation'],
    'Farama Foundation': ['Farama-Foundation'],
    'The Fintech Open Source Foundation': ['finos'],
    'Community Ox': ['Community-Ox'],
    'Commonhaus': ['commonhaus']
}

# Properties to extract from the repository JSON object, as you requested.
prop_list = [
    'html_url', 'fork', 'created_at', 'updated_at', 'pushed_at', 'git_url',
    'size', 'stargazers_count', 'watchers_count', 'language', 'forks_count',
    'archived', 'disabled', 'open_issues_count', 'license', 'allow_forking'
]

# --- Main Script ---

def get_repo_data(company, path):
    """Fetches repository data for a given GitHub path (org, user, or repo)."""
    repo_list = []

    # Determine the API URL and entity type
    if '/' in path: # Case 1: Specific repository (e.g., 'facebook/react')
        api_url = f'https://api.github.com/repos/{path}'
        entity_type = 'repo'
    else: # Case 2: Organization or User
        # First, check if it's an organization
        org_check_response = requests.get(f'https://api.github.com/orgs/{path}', headers=headers)
        if org_check_response.status_code == 200:
            api_url = f'https://api.github.com/orgs/{path}/repos'
            entity_type = 'org'
        else: # If not an org, assume it's a user
            api_url = f'https://api.github.com/users/{path}/repos'
            entity_type = 'user'

    # Paginate through repository results
    page = 1
    while True:
        params = {'per_page': 100, 'page': page}
        try:
            response = requests.get(api_url, headers=headers, params=params)

            # Handle rate limiting
            if response.status_code == 403 and 'rate limit exceeded' in response.text.lower():
                reset_time = int(response.headers.get('X-RateLimit-Reset', time.time()))
                wait_time = max(reset_time - time.time(), 0) + 5
                tqdm.write(f"Rate limit exceeded. Waiting {wait_time:.0f} seconds...")
                time.sleep(wait_time)
                continue # Retry the same request

            if response.status_code != 200:
                tqdm.write(f"Error fetching data for {path}: {response.status_code} - {response.reason}")
                break

            repos = response.json()

            # If it's a specific repo, the response isn't a list, so we wrap it
            if entity_type == 'repo':
                repos = [repos]

            if not repos: # No more repositories to fetch
                break

            for repo in repos:
                # --- Language Filtering Logic ---
                repo_lang = repo.get('language')

                # Check if the repo's language is in our target set.
                # This handles cases where language is None or a string.
                lang_to_check = repo_lang.lower() if repo_lang else None
                if lang_to_check not in target_languages_set:
                    continue # Skip this repo if its language is not in our target list

                # --- Process and store the repo data ---
                repo_properties = {'Organization': company}
                for prop in prop_list:
                    prop_value = repo.get(prop)
                    # The 'license' field is a nested dictionary, so we extract the name
                    if prop == "license" and prop_value is not None:
                        repo_properties[prop] = prop_value.get('name')
                    else:
                        repo_properties[prop] = prop_value
                repo_list.append(repo_properties)

            # For specific repos or if the returned page is smaller than the requested size, stop.
            if entity_type == 'repo' or len(repos) < 100:
                break

            page += 1

        except requests.exceptions.RequestException as e:
            tqdm.write(f"A network error occurred for {path}: {e}")
            break

    return repo_list

# --- Execution ---
all_repos_data = []

# Use tqdm for a nice progress bar
for company, paths in tqdm(org_urls.items(), desc="Processing Organizations"):
    for path in paths:
        tqdm.write(f"Fetching repos for: {company} -> {path}")
        company_repos = get_repo_data(company, path)
        if company_repos:
            all_repos_data.extend(company_repos)
            tqdm.write(f"-> Found and kept {len(company_repos)} repositories for {path} (matching target languages)")

# Create the final DataFrame
df = pd.DataFrame(all_repos_data)

# Ensure the DataFrame is not empty before proceeding
if not df.empty:
    # Add the 'Organization' column to the final list of columns for the DataFrame
    final_columns = ['Organization'] + prop_list
    df = df[final_columns]

print(f"\nTotal repositories saved (after filtering for target languages): {len(df)}")

if not df.empty:
    print("\nSample of the collected data:")
    print(df.head())

# Save the DataFrame to a CSV file
output_filename = "github_foundations_repos_filtered.csv"
df.to_csv(output_filename, index=False)
print(f"\nData successfully saved to {output_filename}")

Processing Organizations:   0%|          | 0/32 [00:00<?, ?it/s]

Fetching repos for: Apache -> apache


Processing Organizations:   3%|▎         | 1/32 [00:52<27:10, 52.61s/it]

-> Found and kept 1759 repositories for apache (matching target languages)
Fetching repos for: Mozilla -> mozilla


Processing Organizations:   6%|▋         | 2/32 [01:38<24:14, 48.50s/it]

...

-> Found and kept 100 repositories for finos (matching target languages)
Fetching repos for: Community Ox -> Community-Ox


Processing Organizations:  97%|█████████▋| 31/32 [04:07<00:01,  1.42s/it]

Error fetching data for Community-Ox: 404 - Not Found
Fetching repos for: Commonhaus -> commonhaus


Processing Organizations: 100%|██████████| 32/32 [04:08<00:00,  7.75s/it]


-> Found and kept 4 repositories for commonhaus (matching target languages)

Total repositories saved (after filtering for target languages): 6431

Sample of the collected data:
  Organization                             html_url   fork  \
0       Apache  https://github.com/apache/tapestry3  False   
1       Apache  https://github.com/apache/apr-iconv  False   
2       Apache  https://github.com/apache/tapestry4  False   
3       Apache    https://github.com/apache/xalan-j  False   
4       Apache       https://github.com/apache/etch  False   

             created_at            updated_at             pushed_at  \
0  2009-03-27T15:41:52Z  2025-04-14T03:46:39Z  2024-12-08T15:49:38Z   
1  2009-03-27T15:41:52Z  2025-06-03T03:02:22Z  2019-01-01T11:45:15Z   
2  2009-03-27T15:41:53Z  2025-06-19T19:37:14Z  2024-12-08T15:51:22Z   
3  2009-03-27T15:41:55Z  2025-07-02T10:13:48Z  2022-10-24T18:27:46Z   
4  2009-03-27T15:41:55Z  2022-11-28T16:04:48Z  2017-04-28T20:27:41Z   

                      